In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit1025.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit1047.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit486.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit728.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit1013.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit282.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit687.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit889.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit554.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit772.txt
/kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels/circuit305.txt
/kaggle/input/yol

In [1]:
!pip install ultralytics --quiet

import os, time, pandas as pd, torch, gc
from ultralytics import YOLO


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 99.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.2 MB/s eta 0:00:00:00:0100:01
Creating new Ultralyti

In [2]:
# ===============================
# Kaggle Notebook: YOLOv8 Ablation Study
# ===============================


# -------------------------------
# Dataset path
# -------------------------------
data_yaml = '/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml'

# -------------------------------
# Base Parameters (default setup)
# -------------------------------
base_params = {
    "epochs": 32,
    "batch": 32,
    "lr0": 0.01,
    "weight_decay": 0.001,
    "optimizer": "SGD",
    "imgsz": 640,
    "dropout": 0.0,
    "freeze": 0,
    "activation": "SiLU"
}

# -------------------------------
# Initialize Results File
# -------------------------------
results_file = "ablation_results.csv"
if not os.path.exists(results_file):
    pd.DataFrame(columns=[
        "Case", "Experiment No", "Experiment Name",
        "Epochs", "Batch", "lr0", "Weight Decay", "Optimizer",
        "Image Size", "Dropout", "Freeze Layers", "Activation",
        "Training Time (s)", "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"
    ]).to_csv(results_file, index=False)

# -------------------------------
# Logging Function
# -------------------------------
def log_result(case, exp_no, exp_name, params, results, train_time, error=False):
    df = pd.read_csv(results_file)

    if error or results is None:
        new_row = {
            "Case": case,
            "Experiment No": exp_no,
            "Experiment Name": exp_name,
            "Epochs": params["epochs"],
            "Batch": params["batch"],
            "lr0": params["lr0"],
            "Weight Decay": params["weight_decay"],
            "Optimizer": params["optimizer"],
            "Image Size": params["imgsz"],
            "Dropout": params["dropout"],
            "Freeze Layers": params["freeze"],
            "Activation": params["activation"],
            "Training Time (s)": "Error",
            "Precision": "Error",
            "Recall": "Error",
            "mAP@0.5": "Error",
            "mAP@0.5:0.95": "Error",
        }
    else:
        new_row = {
            "Case": case,
            "Experiment No": exp_no,
            "Experiment Name": exp_name,
            "Epochs": params["epochs"],
            "Batch": params["batch"],
            "lr0": params["lr0"],
            "Weight Decay": params["weight_decay"],
            "Optimizer": params["optimizer"],
            "Image Size": params["imgsz"],
            "Dropout": params["dropout"],
            "Freeze Layers": params["freeze"],
            "Activation": params["activation"],
            "Training Time (s)": round(train_time, 2),
            "Precision": results.results_dict["metrics/precision(B)"],
            "Recall": results.results_dict["metrics/recall(B)"],
            "mAP@0.5": results.results_dict["metrics/mAP50(B)"],
            "mAP@0.5:0.95": results.results_dict["metrics/mAP50-95(B)"],
        }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(results_file, index=False)

# -------------------------------
# Run Ablation Function
# -------------------------------
def run_ablation(case_name, param_name, param_values):
    """
    Run experiments by varying param_name over param_values.
    """
    for i, val in enumerate(param_values, start=1):
        # copy base params
        params = base_params.copy()
        params[param_name] = val

        print(f"\n🚀 {case_name} | Experiment {i}: {param_name}={val}")
        gc.collect(); torch.cuda.empty_cache()
        model = YOLO("yolov8m.pt")

        start_time = time.time()
        try:
            results = model.train(
                data=data_yaml,
                epochs=params["epochs"],
                batch=params["batch"],
                lr0=params["lr0"],
                optimizer=params["optimizer"],
                imgsz=params["imgsz"],
                weight_decay=params["weight_decay"],
                dropout=params["dropout"],   # only works if dropout is supported
                freeze=params["freeze"]
            )
            end_time = time.time()
            log_result(case_name, i, f"{param_name}_{val}", params, results, end_time - start_time)

        except Exception as e:
            print(f"❌ Error in {param_name}={val}: {e}")
            log_result(case_name, i, f"{param_name}_{val}", params, None, 0, error=True)

    # show final table
    df = pd.read_csv(results_file)
    display(df)




In [4]:
# Case 4: Varying Dropout
run_ablation("Case 4: Dropout", "dropout", [0, 0.1, 0.2, 0.3, 0.5])


🚀 Case 4: Dropout | Experiment 1: dropout=0


Ultralytics 8.3.185 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=100, perspe

Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 6.2±5.3 MB/s, size: 51.9 KB)


train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:06<00:00, 145.16it/s]


...

                   all        230       1440      0.991      0.989      0.988      0.691



32 epochs completed in 0.318 hours.
Optimizer stripped from runs/detect/train5/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train5/weights/best.pt, 52.0MB

Validating runs/detect/train5/weights/best.pt...
Ultralytics 8.3.185 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 92 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:03<00:00,  1.00it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.991      0.991      0.987      0.692
     DC Voltage Source        103        144          1      0.979      0.984       0.76
              Resistor        183        605      0.985      0.992      0.987      0.649
        Current Source        113        150      0.982          1      0.984      0.739
              Inductor        117        177      0.994      0.988       0.99      0.647
             Capacitor        115        184      0.993      0.989      0.985      0.639
     AC Voltage Source         65        180      0.993      0.994      0.995      0.717
Speed: 0.1ms preprocess, 8.2ms inference, 0.0ms loss, 2.8ms postprocess per image
Results saved to runs/detect/train5


,Case,Experiment No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 4: Dropout,1,dropout_0,32,32,0.01,0.001,SGD,640,0.0,0,SiLU,1176.32,0.991306,0.990508,0.987335,0.69197
1,Case 4: Dropout,2,dropout_0.1,32,32,0.01,0.001,SGD,640,0.1,0,SiLU,1161.46,0.991306,0.990508,0.987335,0.69197
2,Case 4: Dropout,3,dropout_0.2,32,32,0.01,0.001,SGD,640,0.2,0,SiLU,1161.13,0.991306,0.990508,0.987335,0.69197
3,Case 4: Dropout,4,dropout_0.3,32,32,0.01,0.001,SGD,640,0.3,0,SiLU,1160.67,0.991306,0.990508,0.987335,0.69197
4,Case 4: Dropout,5,dropout_0.5,32,32,0.01,0.001,SGD,640,0.5,0,SiLU,1164.58,0.991306,0.990508,0.987335,0.69197


In [5]:
# Case 6: Different Optimizers
run_ablation("Case 6: Optimizer", "optimizer", ["SGD", "Adam", "AdamW", "RMSProp", "Adamax"])


🚀 Case 6: Optimizer | Experiment 1: optimizer=SGD
Ultralytics 8.3.185 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train6, nbs=64, nms=False, opset=None, optimize=False, 

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 505.92it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 31.3±9.7 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 341.58it/s]

...

                   all        230       1440      0.989      0.985      0.988      0.691



32 epochs completed in 0.320 hours.
Optimizer stripped from runs/detect/train10/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train10/weights/best.pt, 52.0MB

Validating runs/detect/train10/weights/best.pt...
Ultralytics 8.3.185 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 92 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:04<00:00,  1.05s/it]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.989      0.987      0.988      0.691
     DC Voltage Source        103        144      0.993      0.976      0.985      0.762
              Resistor        183        605      0.984      0.991      0.983      0.649
        Current Source        113        150      0.981      0.993      0.989      0.738
              Inductor        117        177      0.993      0.989      0.989      0.644
             Capacitor        115        184      0.992      0.989       0.99      0.624
     AC Voltage Source         65        180      0.989      0.985      0.994      0.731
Speed: 0.1ms preprocess, 8.5ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to runs/detect/train10


,Case,Experiment No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 4: Dropout,1,dropout_0,32,32,0.01,0.001,SGD,640,0.0,0,SiLU,1176.32,0.991306,0.990508,0.987335,0.691970
1,Case 4: Dropout,2,dropout_0.1,32,32,0.01,0.001,SGD,640,0.1,0,SiLU,1161.46,0.991306,0.990508,0.987335,0.691970
2,Case 4: Dropout,3,dropout_0.2,32,32,0.01,0.001,SGD,640,0.2,0,SiLU,1161.13,0.991306,0.990508,0.987335,0.691970
3,Case 4: Dropout,4,dropout_0.3,32,32,0.01,0.001,SGD,640,0.3,0,SiLU,1160.67,0.991306,0.990508,0.987335,0.691970
4,Case 4: Dropout,5,dropout_0.5,32,32,0.01,0.001,SGD,640,0.5,0,SiLU,1164.58,0.991306,0.990508,0.987335,0.691970
5,Case 6: Optimizer,1,optimizer_SGD,32,32,0.01,0.001,SGD,640,0.0,0,SiLU,1161.23,0.991306,0.990508,0.987335,0.691970
6,Case 6: Optimizer,2,optimizer_Adam,32,32,0.01,0.001,Adam,640,0.0,0,SiLU,1172.61,0.986126,0.986969,0.987008,0.681319
7,Case 6: Optimizer,3,optimizer_AdamW,32,32,0.01,0.001,AdamW,640,0.0,0,SiLU,1168.31,0.986241,0.985495,0.988257,0.687184
8,Case 6: Optimizer,4,optimizer_RMSProp,32,32,0.01,0.001,RMSProp,640,0.0,0,SiLU,1219.09,0.284918,0.290732,0.277995,0.159001
9,Case 6: Optimizer,5,optimizer_Adamax,32,32,0.01,0.001,Adamax,640,0.0,0,SiLU,1171.89,0.988610,0.987190,0.988335,0.691340


In [6]:
# Case 5: Varying Image Size
run_ablation("Case 5: Image Size", "imgsz", [320, 512, 640])


🚀 Case 5: Image Size | Experiment 1: imgsz=320
Ultralytics 8.3.185 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train11, nbs=64, nms=False, opset=None, optimize=False, op

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:01<00:00, 512.76it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 16.0±10.5 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 377.81it/s]

...

                   all        230       1440      0.991      0.989      0.988      0.691



32 epochs completed in 0.316 hours.
Optimizer stripped from runs/detect/train13/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train13/weights/best.pt, 52.0MB

Validating runs/detect/train13/weights/best.pt...
Ultralytics 8.3.185 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 92 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:03<00:00,  1.05it/s]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.991      0.991      0.987      0.692
     DC Voltage Source        103        144          1      0.979      0.984       0.76
              Resistor        183        605      0.985      0.992      0.987      0.649
        Current Source        113        150      0.982          1      0.984      0.739
              Inductor        117        177      0.994      0.988       0.99      0.647
             Capacitor        115        184      0.993      0.989      0.985      0.639
     AC Voltage Source         65        180      0.993      0.994      0.995      0.717
Speed: 0.1ms preprocess, 8.2ms inference, 0.0ms loss, 3.2ms postprocess per image
Results saved to runs/detect/train13


,Case,Experiment No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 4: Dropout,1,dropout_0,32,32,0.01,0.001,SGD,640,0.0,0,SiLU,1176.32,0.991306,0.990508,0.987335,0.691970
1,Case 4: Dropout,2,dropout_0.1,32,32,0.01,0.001,SGD,640,0.1,0,SiLU,1161.46,0.991306,0.990508,0.987335,0.691970
2,Case 4: Dropout,3,dropout_0.2,32,32,0.01,0.001,SGD,640,0.2,0,SiLU,1161.13,0.991306,0.990508,0.987335,0.691970
3,Case 4: Dropout,4,dropout_0.3,32,32,0.01,0.001,SGD,640,0.3,0,SiLU,1160.67,0.991306,0.990508,0.987335,0.691970
4,Case 4: Dropout,5,dropout_0.5,32,32,0.01,0.001,SGD,640,0.5,0,SiLU,1164.58,0.991306,0.990508,0.987335,0.691970
5,Case 6: Optimizer,1,optimizer_SGD,32,32,0.01,0.001,SGD,640,0.0,0,SiLU,1161.23,0.991306,0.990508,0.987335,0.691970
6,Case 6: Optimizer,2,optimizer_Adam,32,32,0.01,0.001,Adam,640,0.0,0,SiLU,1172.61,0.986126,0.986969,0.987008,0.681319
7,Case 6: Optimizer,3,optimizer_AdamW,32,32,0.01,0.001,AdamW,640,0.0,0,SiLU,1168.31,0.986241,0.985495,0.988257,0.687184
8,Case 6: Optimizer,4,optimizer_RMSProp,32,32,0.01,0.001,RMSProp,640,0.0,0,SiLU,1219.09,0.284918,0.290732,0.277995,0.159001
9,Case 6: Optimizer,5,optimizer_Adamax,32,32,0.01,0.001,Adamax,640,0.0,0,SiLU,1171.89,0.988610,0.987190,0.988335,0.691340


In [ ]:
# Case 8: Freeze Layers
run_ablation("Case 8: Freeze Layers", "freeze", [0, 10, 15, 22])


🚀 Case 8: Freeze Layers | Experiment 1: freeze=0
Ultralytics 8.3.185 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train14, nbs=64, nms=False, opset=None, optimize=False, 

train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:02<00:00, 434.15it/s]

WARNING ⚠️ train: Cache directory /kaggle/input/yolo-component-data/YOLO_component-dataset/train is not writeable, cache not saved.
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 22.7±12.4 MB/s, size: 35.4 KB)


val: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/val/labels... 230 images, 0 backgrounds, 0 corrupt: 100%|██████████| 230/230 [00:00<00:00, 282.31it/s]

...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       3/32      7.68G      1.359      1.005       1.22        203        640: 100%|██████████| 29/29 [00:16<00:00,  1.76it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:02<00:00,  1.62it/s]

                   all        230       1440      0.939       0.96      0.978       0.63



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       4/32      7.68G      1.351     0.8901        1.2        241        640:  69%|██████▉   | 20/29 [00:11<00:05,  1.75it/s]

In [4]:
# Case 8: Freeze Layers
run_ablation("Case 8: Freeze Layers", "freeze", [15, 22])


🚀 Case 8: Freeze Layers | Experiment 1: freeze=15


Ultralytics 8.3.185 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=15, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=100, per

Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [96, 192, 3, 2]               
  4                  -1  4    813312  ultralytics.nn.modules.block.C2f             [192, 192, 4, True]           
  5                  -1  1    664320  ultralytics.nn.modules.conv.Conv             [192, 384, 3, 2]              
  6                  -1  4   3248640  ultralytics.nn.modules.block.C2f             [384, 384, 4, True]           
  7                  -1  1   1991808  ultralytics

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 7.1±6.2 MB/s, size: 51.9 KB)


train: Scanning /kaggle/input/yolo-component-data/YOLO_component-dataset/train/labels... 919 images, 0 backgrounds, 0 corrupt: 100%|██████████| 919/919 [00:06<00:00, 147.82it/s]


...

                   all        230       1440       0.98      0.978      0.985       0.64



32 epochs completed in 0.134 hours.
Optimizer stripped from runs/detect/train2/weights/last.pt, 52.0MB
Optimizer stripped from runs/detect/train2/weights/best.pt, 52.0MB

Validating runs/detect/train2/weights/best.pt...
Ultralytics 8.3.185 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 92 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 4/4 [00:04<00:00,  1.02s/it]
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.978      0.977      0.984      0.645
     DC Voltage Source        103        144      0.994      0.972       0.98      0.715
              Resistor        183        605      0.983       0.98      0.987      0.623
        Current Source        113        150       0.96       0.98      0.984      0.691
              Inductor        117        177      0.978      0.983       0.98      0.577
             Capacitor        115        184      0.978      0.973      0.982      0.549
     AC Voltage Source         65        180      0.978      0.973      0.993      0.714
Speed: 0.1ms preprocess, 8.2ms inference, 0.0ms loss, 2.9ms postprocess per image
Results saved to runs/detect/train2


,Case,Experiment No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 8: Freeze Layers,1,freeze_15,32,32,0.01,0.001,SGD,640,0.0,15,SiLU,675.82,0.988366,0.986431,0.987107,0.689567
1,Case 8: Freeze Layers,2,freeze_22,32,32,0.01,0.001,SGD,640,0.0,22,SiLU,504.21,0.978447,0.976879,0.984360,0.644947


In [3]:
# Case 3: Varying Batch Size
# ===============================
run_ablation("Case 3: Batch Size", "batch", [16, 32, 64])


🚀 Case 3: Batch Size | Experiment 1: batch=16
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optim

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440       0.99      0.989      0.987      0.687
     DC Voltage Source        103        144      0.992      0.979      0.984      0.759
              Resistor        183        605      0.985      0.994      0.986      0.649
        Current Source        113        150      0.985          1      0.988      0.738
              Inductor        117        177      0.989      0.989      0.986      0.642
             Capacitor        115        184      0.993      0.989      0.986      0.611
     AC Voltage Source         65        180      0.994      0.986      0.993      0.721
Speed: 0.1ms preprocess, 9.0ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /kaggle/working/runs/detect/train

🚀 Case 3: Batch Size | Experiment 2: batch=32
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.991      0.991      0.987      0.692
     DC Voltage Source        103        144          1      0.979      0.984       0.76
              Resistor        183        605      0.985      0.992      0.987       0.65
        Current Source        113        150      0.982          1      0.984       0.74
              Inductor        117        177      0.994      0.988       0.99      0.646
             Capacitor        115        184      0.993      0.989      0.985      0.638
     AC Voltage Source         65        180      0.993      0.994      0.995      0.717
Speed: 0.1ms preprocess, 8.1ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/train2

🚀 Case 3: Batch Size | Experiment 3: batch=64
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=

,Case,Experiment No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 3: Batch Size,1,batch_16,32,16,0.01,0.001,SGD,640,0.0,0,SiLU,1267.75,0.9898094080557646,0.9894705611842188,0.9873951921114864,0.6866352229419413
1,Case 3: Batch Size,2,batch_32,32,32,0.01,0.001,SGD,640,0.0,0,SiLU,1153.51,0.991306588627882,0.990504688006188,0.9873156486165096,0.6919208489023024
2,Case 3: Batch Size,3,batch_64,32,64,0.01,0.001,SGD,640,0.0,0,SiLU,Error,Error,Error,Error,Error


In [4]:
run_ablation("Case 7: Learning Rate", "lr0", [0.001, 0.005, 0.01, 0.05, 0.1])


🚀 Case 7: Learning Rate | Experiment 1: lr0=0.001
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optimize=False,

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.987       0.99      0.988      0.689
     DC Voltage Source        103        144      0.986      0.979      0.981      0.757
              Resistor        183        605      0.985      0.989      0.987      0.649
        Current Source        113        150      0.979          1      0.986      0.744
              Inductor        117        177      0.985      0.989      0.992      0.644
             Capacitor        115        184      0.989      0.989       0.99      0.626
     AC Voltage Source         65        180      0.994      0.992      0.994      0.713
Speed: 0.1ms preprocess, 8.1ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /kaggle/working/runs/detect/train4

🚀 Case 7: Learning Rate | Experiment 2: lr0=0.005
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, 

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440       0.99       0.99      0.988      0.697
     DC Voltage Source        103        144          1      0.979      0.986      0.777
              Resistor        183        605      0.985      0.993       0.99      0.652
        Current Source        113        150       0.98      0.998      0.982      0.733
              Inductor        117        177      0.991      0.989      0.987      0.641
             Capacitor        115        184      0.992      0.989      0.986      0.651
     AC Voltage Source         65        180      0.992      0.994      0.994      0.725
Speed: 0.1ms preprocess, 8.2ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/train5

🚀 Case 7: Learning Rate | Experiment 3: lr0=0.01
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, b

...

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.987      0.987      0.989      0.695
     DC Voltage Source        103        144      0.993      0.969      0.984      0.761
              Resistor        183        605      0.982       0.99      0.986      0.654
        Current Source        113        150      0.979          1      0.989      0.741
              Inductor        117        177      0.992      0.989      0.994      0.659
             Capacitor        115        184      0.983      0.989      0.991      0.628
     AC Voltage Source         65        180      0.994      0.985      0.989      0.725
Speed: 0.1ms preprocess, 8.2ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /kaggle/working/runs/detect/train7

🚀 Case 7: Learning Rate | Experiment 5: lr0=0.1
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bg

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.984      0.981      0.989      0.693
     DC Voltage Source        103        144          1      0.952      0.985      0.758
              Resistor        183        605      0.983      0.988      0.989      0.656
        Current Source        113        150      0.946      0.993      0.985      0.734
              Inductor        117        177      0.993      0.989      0.993      0.653
             Capacitor        115        184      0.995      0.987       0.99       0.64
     AC Voltage Source         65        180      0.989      0.978      0.991      0.719
Speed: 0.1ms preprocess, 8.1ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /kaggle/working/runs/detect/train8


,Case,Experiment No,Experiment Name,Epochs,Batch,lr0,Weight Decay,Optimizer,Image Size,Dropout,Freeze Layers,Activation,Training Time (s),Precision,Recall,mAP@0.5,mAP@0.5:0.95
0,Case 3: Batch Size,1,batch_16,32,16,0.010,0.001,SGD,640,0.0,0,SiLU,1267.75,0.9898094080557646,0.9894705611842188,0.9873951921114864,0.6866352229419413
1,Case 3: Batch Size,2,batch_32,32,32,0.010,0.001,SGD,640,0.0,0,SiLU,1153.51,0.991306588627882,0.990504688006188,0.9873156486165096,0.6919208489023024
2,Case 3: Batch Size,3,batch_64,32,64,0.010,0.001,SGD,640,0.0,0,SiLU,Error,Error,Error,Error,Error
3,Case 7: Learning Rate,1,lr0_0.001,32,32,0.001,0.001,SGD,640,0.0,0,SiLU,1157.25,0.9865793731082227,0.9895758357409906,0.9881686723190005,0.6887520150251867
4,Case 7: Learning Rate,2,lr0_0.005,32,32,0.005,0.001,SGD,640,0.0,0,SiLU,1157.34,0.9901062556882234,0.9903801637215488,0.9875037459218959,0.696559365054343
5,Case 7: Learning Rate,3,lr0_0.01,32,32,0.010,0.001,SGD,640,0.0,0,SiLU,1157.42,0.9913065886278819,0.990504688006188,0.9873156486165097,0.6919208489023024
6,Case 7: Learning Rate,4,lr0_0.05,32,32,0.050,0.001,SGD,640,0.0,0,SiLU,1158.61,0.9871818691472248,0.9870353716572411,0.9888865121722459,0.6946677077190417
7,Case 7: Learning Rate,5,lr0_0.1,32,32,0.100,0.001,SGD,640,0.0,0,SiLU,1168.65,0.9843538302233757,0.98129790914341,0.9889075867339837,0.6934032622745152


In [6]:
import yaml
import shutil

def create_model_yaml(base_yaml, new_yaml, activation):
    """
    Create a new model YAML file with a custom activation function.
    """
    with open(base_yaml, "r") as f:
        cfg = yaml.safe_load(f)

    # Replace activation everywhere it appears
    if "backbone" in cfg:
        for layer in cfg["backbone"]:
            if isinstance(layer, list) and len(layer) > 0 and isinstance(layer[-1], dict):
                if "activation" in layer[-1]:
                    layer[-1]["activation"] = activation
    if "head" in cfg:
        for layer in cfg["head"]:
            if isinstance(layer, list) and len(layer) > 0 and isinstance(layer[-1], dict):
                if "activation" in layer[-1]:
                    layer[-1]["activation"] = activation

    # Save new yaml
    with open(new_yaml, "w") as f:
        yaml.safe_dump(cfg, f)


In [7]:
# ===============================
# Case 8: Activation Functions
# ===============================
activations = ["ReLU", "Mish", "LeakyReLU", "SiLU"]

for i, act in enumerate(activations, start=1):
    model_yaml = f"yolov8m_{act}.yaml"
    create_model_yaml("/kaggle/working/yolov8m.yaml", model_yaml, act)

    print(f"\n🚀 Case 8 | Experiment {i}: Activation={act}")
    gc.collect(); torch.cuda.empty_cache()
    model = YOLO(model_yaml)  # use custom YAML with new activation

    start_time = time.time()
    try:
        results = model.train(
            data=data_yaml,
            epochs=base_params["epochs"],
            batch=base_params["batch"],
            lr0=base_params["lr0"],
            optimizer=base_params["optimizer"],
            imgsz=base_params["imgsz"],
            weight_decay=base_params["weight_decay"],
            dropout=base_params["dropout"],
            freeze=base_params["freeze"]
        )
        end_time = time.time()
        log_result("Case 8: Activation", i, f"activation_{act}", 
                   {**base_params, "activation": act}, results, end_time - start_time)

    except Exception as e:
        print(f"❌ Error in Activation={act}: {e}")
        log_result("Case 8: Activation", i, f"activation_{act}", 
                   {**base_params, "activation": act}, None, 0, error=True)

# show results
df = pd.read_csv(results_file)
display(df)


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/yolov8m.yaml'

In [10]:
# ===============================
# Case 9: Activation Functions
# ===============================

import yaml, copy, os, time, gc
import pandas as pd
from ultralytics import YOLO
from ultralytics.nn.tasks import yaml_model_load
from tabulate import tabulate

# -------------------------------
# Results file
# -------------------------------
results_file = "case9_results.csv"
if not os.path.exists(results_file):
    pd.DataFrame(columns=[
        "Experiment", "Epochs", "Batch Size", "Optimizer", "Image Size", "Learning Rate",
        "Activation", "Freeze Layers", "Training Time (s)",
        "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"
    ]).to_csv(results_file, index=False)

# -------------------------------
# Dataset
# -------------------------------
data_yaml = "/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml"

# -------------------------------
# Baseline Params
# -------------------------------
base_params = {
    "epochs": 32,
    "batch": 32,
    "optimizer": "AdamW",      # best from earlier cases
    "lr0": 0.001,              # best from Case 7
    "weight_decay": 0.001,     # best from Case 2
    "dropout": 0.0,
    "imgsz": 640,
    "freeze": 0
}

# -------------------------------
# Load Base Config
# -------------------------------
cfg = yaml_model_load("yolov8m.yaml")  # you can switch to yolov8n.yaml or yolov8m.yaml

def create_yaml_with_activation(cfg, act_expr, base_name="yolov8s"):
    """
    Create a new YAML file with the given activation function.
    """
    cfg_mod = copy.deepcopy(cfg)
    cfg_mod['activation'] = act_expr

    safe_name = act_expr.replace("nn.","").replace("(","_").replace(")","").replace(".","")
    out_path = f"/kaggle/working/{base_name}_{safe_name.lower()}.yaml"

    with open(out_path, "w") as f:
        yaml.dump(cfg_mod, f)

    print(f"✅ Created {out_path}")
    return out_path

# -------------------------------
# Define Activations
# -------------------------------
activations = {
    "ReLU": create_yaml_with_activation(cfg, "nn.ReLU()"),
    "LeakyReLU": create_yaml_with_activation(cfg, "nn.LeakyReLU(0.1)"),
    "Mish": create_yaml_with_activation(cfg, "nn.Mish()"),
    "SiLU": create_yaml_with_activation(cfg, "nn.SiLU()")  # baseline
}

# -------------------------------
# Run Case 9
# -------------------------------
all_results = []

for act, yaml_path in activations.items():
    print(f"\n🚀 Training with {act} activation...")
    gc.collect(); 
    try:
        model = YOLO(yaml_path)

        start = time.time()
        results = model.train(
            data=data_yaml,
            epochs=base_params["epochs"],
            batch=base_params["batch"],
            lr0=base_params["lr0"],
            optimizer=base_params["optimizer"],
            imgsz=base_params["imgsz"],
            weight_decay=base_params["weight_decay"],
            dropout=base_params["dropout"],
            freeze=base_params["freeze"]
        )
        end = time.time()

        # Validate model after training
        val = model.val()
        metrics = val.results_dict

        result = {
            "Experiment": f"case9_act_{act}",
            "Epochs": base_params["epochs"],
            "Batch Size": base_params["batch"],
            "Optimizer": base_params["optimizer"],
            "Image Size": base_params["imgsz"],
            "Learning Rate": base_params["lr0"],
            "Activation": act,
            "Freeze Layers": base_params["freeze"],
            "Training Time (s)": round(end - start, 2),
            "Precision": metrics.get("metrics/precision(B)", metrics.get("metrics/precision", "N/A")),
            "Recall": metrics.get("metrics/recall(B)", metrics.get("metrics/recall", "N/A")),
            "mAP@0.5": metrics.get("metrics/mAP50(B)", metrics.get("metrics/mAP_0.5", "N/A")),
            "mAP@0.5:0.95": metrics.get("metrics/mAP50-95(B)", metrics.get("metrics/mAP_0.5:0.95", "N/A"))
        }

    except Exception as e:
        print(f"❌ Error training with {act}: {e}")
        result = {
            "Experiment": f"case9_act_{act}",
            "Epochs": base_params["epochs"],
            "Batch Size": base_params["batch"],
            "Optimizer": base_params["optimizer"],
            "Image Size": base_params["imgsz"],
            "Learning Rate": base_params["lr0"],
            "Activation": act,
            "Freeze Layers": base_params["freeze"],
            "Training Time (s)": "Error",
            "Precision": "Error",
            "Recall": "Error",
            "mAP@0.5": "Error",
            "mAP@0.5:0.95": "Error"
        }

    # Append to CSV
    df = pd.read_csv(results_file)
    df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
    df.to_csv(results_file, index=False)

    all_results.append(result)

# -------------------------------
# Show Final Results
# -------------------------------
print("\n📊 Case 9 Results Table:\n")
print(tabulate(pd.DataFrame(all_results), headers="keys", tablefmt="grid"))


✅ Created /kaggle/working/yolov8s_relu_.yaml
✅ Created /kaggle/working/yolov8s_leakyrelu_01.yaml
✅ Created /kaggle/working/yolov8s_mish_.yaml
✅ Created /kaggle/working/yolov8s_silu_.yaml

🚀 Training with ReLU activation...
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, ma

KeyboardInterrupt: 

In [11]:
# ===============================
# Case 9: Activation Functions (YOLOv8m)
# ===============================

import yaml, copy, os, time, gc
import pandas as pd
from ultralytics import YOLO
from ultralytics.nn.tasks import yaml_model_load
from tabulate import tabulate

# -------------------------------
# Results file
# -------------------------------
results_file = "case9_results.csv"
if not os.path.exists(results_file):
    pd.DataFrame(columns=[
        "Experiment", "Epochs", "Batch Size", "Optimizer", "Image Size", "Learning Rate",
        "Activation", "Freeze Layers", "Training Time (s)",
        "Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"
    ]).to_csv(results_file, index=False)

# -------------------------------
# Dataset
# -------------------------------
data_yaml = "/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml"

# -------------------------------
# Baseline Params
# -------------------------------
base_params = {
    "epochs": 32,
    "batch": 32,               # smaller batch to avoid OOM on YOLOv8m
    "optimizer": "SGD",
    "lr0": 0.005,
    "weight_decay": 0.001,
    "dropout": 0.0,
    "imgsz": 640,
    "freeze": 0
}

# -------------------------------
# Load Base Config for YOLOv8m
# -------------------------------
cfg = yaml_model_load("yolov8m.yaml")

def create_yaml_with_activation(cfg, act_expr, base_name="yolov8m"):
    """
    Create a new YAML file with the given activation function.
    """
    cfg_mod = copy.deepcopy(cfg)
    cfg_mod['activation'] = act_expr

    safe_name = act_expr.replace("nn.","").replace("(","_").replace(")","").replace(".","")
    out_path = f"/kaggle/working/{base_name}_{safe_name.lower()}.yaml"

    with open(out_path, "w") as f:
        yaml.dump(cfg_mod, f)

    print(f"✅ Created {out_path}")
    return out_path

# -------------------------------
# Define Activations
# -------------------------------
activations = {
    "ReLU": create_yaml_with_activation(cfg, "nn.ReLU()", base_name="yolov8m"),
    "LeakyReLU": create_yaml_with_activation(cfg, "nn.LeakyReLU(0.1)", base_name="yolov8m"),
    "Mish": create_yaml_with_activation(cfg, "nn.Mish()", base_name="yolov8m"),
    "SiLU": create_yaml_with_activation(cfg, "nn.SiLU()", base_name="yolov8m")  # baseline
}

# -------------------------------
# Run Case 9
# -------------------------------
all_results = []

for act, yaml_path in activations.items():
    print(f"\n🚀 Training with {act} activation...")
    gc.collect()
    try:
        model = YOLO(yaml_path)

        start = time.time()
        results = model.train(
            data=data_yaml,
            epochs=base_params["epochs"],
            batch=base_params["batch"],
            lr0=base_params["lr0"],
            optimizer=base_params["optimizer"],
            imgsz=base_params["imgsz"],
            weight_decay=base_params["weight_decay"],
            dropout=base_params["dropout"],
            freeze=base_params["freeze"]
        )
        end = time.time()

        # Validate model after training
        val = model.val()
        metrics = val.results_dict

        result = {
            "Experiment": f"case9_act_{act}",
            "Epochs": base_params["epochs"],
            "Batch Size": base_params["batch"],
            "Optimizer": base_params["optimizer"],
            "Image Size": base_params["imgsz"],
            "Learning Rate": base_params["lr0"],
            "Activation": act,
            "Freeze Layers": base_params["freeze"],
            "Training Time (s)": round(end - start, 2),
            "Precision": metrics.get("metrics/precision(B)", metrics.get("metrics/precision", "N/A")),
            "Recall": metrics.get("metrics/recall(B)", metrics.get("metrics/recall", "N/A")),
            "mAP@0.5": metrics.get("metrics/mAP50(B)", metrics.get("metrics/mAP_0.5", "N/A")),
            "mAP@0.5:0.95": metrics.get("metrics/mAP50-95(B)", metrics.get("metrics/mAP_0.5:0.95", "N/A"))
        }

    except Exception as e:
        print(f"❌ Error training with {act}: {e}")
        result = {
            "Experiment": f"case9_act_{act}",
            "Epochs": base_params["epochs"],
            "Batch Size": base_params["batch"],
            "Optimizer": base_params["optimizer"],
            "Image Size": base_params["imgsz"],
            "Learning Rate": base_params["lr0"],
            "Activation": act,
            "Freeze Layers": base_params["freeze"],
            "Training Time (s)": "Error",
            "Precision": "Error",
            "Recall": "Error",
            "mAP@0.5": "Error",
            "mAP@0.5:0.95": "Error"
        }

    # Append to CSV
    df = pd.read_csv(results_file)
    df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)
    df.to_csv(results_file, index=False)

    all_results.append(result)

# -------------------------------
# Show Final Results
# -------------------------------
print("\n📊 Case 9 Results Table:\n")
print(tabulate(pd.DataFrame(all_results), headers="keys", tablefmt="grid"))


✅ Created /kaggle/working/yolov8m_relu_.yaml
✅ Created /kaggle/working/yolov8m_leakyrelu_01.yaml
✅ Created /kaggle/working/yolov8m_mish_.yaml
✅ Created /kaggle/working/yolov8m_silu_.yaml

🚀 Training with ReLU activation...
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/yolo-component-data/YOLO_component-dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=32, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, ma

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.971       0.98      0.985      0.663
     DC Voltage Source        103        144      0.942      0.972      0.978      0.741
              Resistor        183        605      0.979      0.985      0.988      0.617
        Current Source        113        150      0.949      0.984       0.98      0.724
              Inductor        117        177      0.977      0.974      0.989      0.617
             Capacitor        115        184      0.993      0.984      0.983       0.57
     AC Voltage Source         65        180      0.989      0.983      0.991      0.705
Speed: 0.1ms preprocess, 8.2ms inference, 0.0ms loss, 2.2ms postprocess per image
Results saved to /kaggle/working/runs/detect/train12
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLOv8m_relu_ summary (fused): 92 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 76.1±51

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.971      0.981      0.985      0.663
     DC Voltage Source        103        144       0.94      0.972      0.978      0.741
              Resistor        183        605      0.979      0.987      0.988      0.618
        Current Source        113        150      0.949      0.986       0.98      0.726
              Inductor        117        177      0.977      0.974      0.989      0.618
             Capacitor        115        184      0.992      0.984      0.983       0.57
     AC Voltage Source         65        180      0.989      0.983      0.991      0.703
Speed: 2.0ms preprocess, 8.1ms inference, 0.0ms loss, 2.5ms postprocess per image
Results saved to /kaggle/working/runs/detect/train122

🚀 Training with LeakyReLU activation...
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0,

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440       0.98      0.976      0.985      0.666
     DC Voltage Source        103        144          1       0.96      0.978      0.753
              Resistor        183        605      0.982      0.975      0.986      0.629
        Current Source        113        150       0.93      0.993      0.978      0.711
              Inductor        117        177      0.977       0.98      0.992      0.607
             Capacitor        115        184      0.994      0.989      0.985      0.591
     AC Voltage Source         65        180      0.994       0.96       0.99      0.708
Speed: 0.1ms preprocess, 8.2ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/train13
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
YOLOv8m_leakyrelu_01 summary (fused): 92 layers, 25,843,234 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        230       1440      0.979      0.976      0.985      0.666
     DC Voltage Source        103        144      0.998      0.958      0.978       0.75
              Resistor        183        605      0.982      0.975      0.986      0.631
        Current Source        113        150       0.93      0.993      0.978       0.71
              Inductor        117        177      0.977       0.98      0.992      0.609
             Capacitor        115        184      0.994      0.989      0.985      0.591
     AC Voltage Source         65        180      0.994      0.959       0.99      0.707
Speed: 1.7ms preprocess, 7.9ms inference, 0.0ms loss, 3.3ms postprocess per image
Results saved to /kaggle/working/runs/detect/train132

🚀 Training with Mish activation...
Ultralytics 8.3.193 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=